In [1]:
from http.client import responses
from pyexpat.errors import messages

from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver #Used for saving messages temprarily to the RAM
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END, START
from typing import Annotated, TypedDict, Literal

In [2]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [3]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [4]:
def chat(state:ChatState):
    messages=state['messages']
    response=model.invoke(messages).content
    return {'messages': [AIMessage(content=response)]}

In [5]:
checkpointer=MemorySaver() #used in saving the previous messages to RAM
graph= StateGraph(ChatState)

graph.add_node("chat",chat)

graph.add_edge(START,"chat")
graph.add_edge("chat",END)

chatbot=graph.compile(checkpointer=checkpointer) # checkpointer=checkpointer used in saving the previous messages to RAM

In [6]:
thread_id="1"   # used in saving the previous messages to RAM

while True:
    user_message= input("Type ypu message here: ")
    if user_message.strip().lower() in ['quit',"bye"]:
        break
    print(user_message)

    response = chatbot.invoke(
    {"messages": [HumanMessage(content=user_message)]},
    config={"configurable": {"thread_id": thread_id}}       #used in saving the previous messages to RAM
    )

    print(response["messages"][-1].content)

Hello

Hello! How can I assist you today?
write a joke on pizza

Sure, here's a light-hearted pizza joke for you:

Why did the pizza go to therapy?

Because it couldn't stop feeling so saucy about itself!


In [7]:
list(chatbot.get_state_history(config={"configurable": {"thread_id": thread_id}} ))

[StateSnapshot(values={'messages': [HumanMessage(content='Hello\n', additional_kwargs={}, response_metadata={}, id='cd968ff0-49d1-4d80-bf23-a2c276b7b396'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, id='99296ff5-22db-4c62-b8de-e75faec892fc', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='write a joke on pizza\n', additional_kwargs={}, response_metadata={}, id='cc39c5a3-320f-4618-ac89-9e7266a8dfa3'), AIMessage(content="Sure, here's a light-hearted pizza joke for you:\n\nWhy did the pizza go to therapy?\n\nBecause it couldn't stop feeling so saucy about itself!", additional_kwargs={}, response_metadata={}, id='bfe715ba-e016-4f37-b705-e6aa54ee3655', tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f132761-b65c-6ffa-8004-f811764e688a'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-04-07T11:37:07.238259+00:00',

ab sb sh lets get a few concepts correct

## Checkpointer
ye kia krta ha ye ap ka main chechpoint nodes ph ap ki states ko save kr daitay ,, ya simple alfaz mai state START sh state 2 ph gaye tu ab state START initial ha wahan ph ap ki state save hui and then state 2 ph apki state save hui,,,, ab eg agr parallel flow ha tu like jis sh shuru ho rha parallel flow lets say state2 wahan ph state save hogi ,,,, and after that jahan ph parallel flows merge ho rhy wahan state save hogi

## Thread_id
thread id basically identify krti ha chat ko k kon si chat k baray mai bt ho rhi like eg chatgpt mai aik sh bht zyada chats hain ab chat 1 ko ap nh thread_id 1 dy di tu wo jb b wo chat continue krni hui you can do that by highlight thread_id =1

## Fault tolerance
is mai kia ha k lets say ap nh state 1 and 2 tu implement kr li and next state/step implement krnty huay error aa gya or ap ki pipeline crash kr gayi ,, tu ab ap chahty ho k ap ki state 0 sh dubara program na start ho rather state 2 onwards ho jahan ph crash aya tha (state3) tu is k liye while invoking

waisy you do

```response = chatbot.invoke(
    {"messages": [HumanMessage(content=user_message)]},
    config={"configurable": {"thread_id": thread_id}}
    )
```
is hs state 0 sh shuru hoga but if you want to continue from crash state simply do

```response = chatbot.invoke(
    None,
    config={"configurable": {"thread_id": thread_id}}
    )
```

as you dont want input you want to contnue from crashed state


# TimeTravel

this is basically a debugging tool k eg mai nh dekha k for thread 1 mughy at a particular state aik input milli thi and us sh agy process hua ab mai dubara dkehna chahta hu k us input ph mughy kia milta ha output mai

tu mai uper jis trh sh state history li mai nh phr checkpoint id li and then onvokde krtay huay wo id b mention ki

In [13]:
response = chatbot.invoke(
    {},  # 👈 empty dict, NOT None
    config={
        "configurable": {
            "thread_id": thread_id,
            "checkpoint_id": "1f132761-91a1-6ca6-8003-34052011087a"
        }
    }
)

for message in response['messages']:
    print(message.content)

Hello

Hello! How can I assist you today?
write a joke on pizza

Sure, here's a light-hearted pizza joke for you:

Why did the pizza go to therapy?

Because it couldn't stop feeling so saucy about itself!


Now as you can see the state upto this was saved
```
Hello

Hello! How can I assist you today?
write a joke on pizza
```

and then generated a seperate joke than the original one